# Smart Equipment Allocation

**MediRent-Ai - Smart Equipment Allocation Module**

This notebook implements the PRD's Smart Equipment Allocation capability as an **intelligent scoring/ranking decision layer**. It does **not** train a separate ML model.

### Objective

For a customer request, recommend the most suitable available equipment unit using:

- Availability
- Equipment condition
- Maintenance status
- Expected return / next scheduled booking
- Requested rental duration
- Current unit utilisation
- Future predicted demand

The notebook uses the existing equipment-utilisation output and the existing demand-prediction output whenever available.

## Important data interpretation

The cleaned dataset contains `Equipment_ID`, so this module can perform **unit-level allocation**.

Some rental rows have missing `Equipment_ID`. Those rows are excluded from unit-level candidate selection because a physical unit cannot be identified reliably.

Maintenance data contains missing values and some historical due dates. Missing maintenance information is treated as **Unknown**, not as proof that an item is safe.

The default setting does not hard-block every overdue historical maintenance record because the dataset's maintenance schedule is incomplete. A unit is hard-filtered when its known maintenance due date falls inside the requested rental period. A strict mode is also provided.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Libraries loaded.")

In [ ]:
# Project path resolution
# Works when the notebook is opened inside the cloned repo, when the repo is
# at /content/MediRent-Ai, and in local/container test environments.

DATA_FILE_NAME = "feature_engineered_medical_equipment_dataset.csv"

candidate_roots = [
    "/content/MediRent-Ai",
    os.getcwd(),
    "/mnt/data",
]

PROJECT_ROOT = None

for root in candidate_roots:
    if os.path.exists(
        os.path.join(
            root,
            "data",
            "processed",
            DATA_FILE_NAME
        )
    ):
        PROJECT_ROOT = root
        break

# Recursive fallback for environments where the notebook execution directory
# is different from the project directory.
if PROJECT_ROOT is None:
    search_roots = ["/content", "/mnt/data"]

    for search_root in search_roots:
        matches = glob.glob(
            os.path.join(
                search_root,
                "**",
                "data",
                "processed",
                DATA_FILE_NAME
            ),
            recursive=True
        )

        if matches:
            PROJECT_ROOT = os.path.dirname(
                os.path.dirname(
                    os.path.dirname(matches[0])
                )
            )
            break

if PROJECT_ROOT is None:
    PROJECT_ROOT = os.getcwd()

DATA_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)

os.makedirs(DATA_DIR, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data:", DATA_DIR)
print("Available processed files:")

for f in sorted(
    glob.glob(
        os.path.join(DATA_DIR, "*.csv")
    )
):
    print(" -", os.path.basename(f))

In [ ]:
# Main input/output paths

CLEANED_PATH = os.path.join(
    DATA_DIR,
    "cleaned_medical_equipment_dataset.csv"
)

FEATURE_PATH = os.path.join(
    DATA_DIR,
    "feature_engineered_medical_equipment_dataset.csv"
)

UTILISATION_PATH = os.path.join(
    DATA_DIR,
    "equipment_utilisation.csv"
)

CATEGORY_UTILISATION_PATH = os.path.join(
    DATA_DIR,
    "category_utilisation.csv"
)

REQUIRED_CLEANED_COLUMNS = [
    "Rental_ID",
    "Equipment",
    "Equipment_Category",
    "Equipment_ID",
    "Rental_Duration_Days",
    "Rental_Status",
    "Equipment_Condition",
    "Booking_Date",
    "Rental_Start_Date",
    "Actual_Return_Date",
    "Expected_Return_Date",
    "Next_Maintenance_Due",
    "Total_Inventory",
]

assert os.path.exists(CLEANED_PATH), f"Missing file: {CLEANED_PATH}"
assert os.path.exists(FEATURE_PATH), f"Missing file: {FEATURE_PATH}"

cleaned = pd.read_csv(CLEANED_PATH)
features = pd.read_csv(FEATURE_PATH)

print("Cleaned dataset shape:", cleaned.shape)
print("Feature-engineered dataset shape:", features.shape)

In [ ]:
# Parse date and numeric columns before building allocation intervals.
DATE_COLUMNS = [
    "Booking_Date",
    "Rental_Start_Date",
    "Actual_Return_Date",
    "Expected_Return_Date",
    "Next_Maintenance_Due",
]

for col in DATE_COLUMNS:
    cleaned[col] = pd.to_datetime(
        cleaned[col],
        errors="coerce"
    )

cleaned["Rental_Duration_Days"] = pd.to_numeric(
    cleaned["Rental_Duration_Days"],
    errors="coerce"
)

cleaned["Total_Inventory"] = pd.to_numeric(
    cleaned["Total_Inventory"],
    errors="coerce"
)

# A cancellation is excluded from availability calculations.
rental_status = (
    cleaned["Rental_Status"]
    .astype(str)
    .str.strip()
    .str.lower()
)

cancellation_status = (
    cleaned.get(
        "Cancellation_Status",
        pd.Series("", index=cleaned.index)
    )
    .astype(str)
    .str.strip()
    .str.lower()
)

cleaned["_Is_Cancelled"] = (
    rental_status.eq("cancelled")
    |
    (
        cancellation_status.str.contains(
            "cancel",
            na=False
        )
        &
        ~cancellation_status.str.contains(
            "not cancel",
            na=False
        )
    )
)

# End date uses actual return when available, expected return otherwise,
# and planned duration as the final fallback.
cleaned["Planned_End_Date"] = (
    cleaned["Rental_Start_Date"]
    + pd.to_timedelta(
        cleaned["Rental_Duration_Days"],
        unit="D"
    )
)

cleaned["End_Date"] = (
    cleaned["Actual_Return_Date"]
    .combine_first(cleaned["Expected_Return_Date"])
    .combine_first(cleaned["Planned_End_Date"])
)

print("Cancelled rows:", int(cleaned["_Is_Cancelled"].sum()))
print("Rows with Equipment_ID:", int(cleaned["Equipment_ID"].notna().sum()))
print("Rows without Equipment_ID:", int(cleaned["Equipment_ID"].isna().sum()))

## 1. Equipment utilisation input

The allocation layer consumes the completed utilisation module rather than training a new utilisation model.

In [ ]:
def derive_utilisation_if_missing(cleaned_df):
    """Fallback only: derives the same pool-level planned-duration metric used by the utilisation notebook."""

    work = cleaned_df[
        (~cleaned_df["_Is_Cancelled"])
        & cleaned_df["Equipment"].notna()
    ].copy()

    analysis_start = work["Rental_Start_Date"].min()
    analysis_end = work["Rental_Start_Date"].max()

    window_days = (
        analysis_end - analysis_start
    ).days + 1

    work["Clip_Start"] = work["Rental_Start_Date"].clip(
        lower=analysis_start,
        upper=analysis_end
    )
    work["Clip_End"] = work["End_Date"].clip(
        lower=analysis_start + pd.Timedelta(days=1),
        upper=analysis_end + pd.Timedelta(days=1)
    )

    work["Planned_Rental_Days_Within"] = (
        work["Clip_End"] - work["Clip_Start"]
    ).dt.days.clip(lower=0)

    rental_days = (
        work.groupby("Equipment")["Planned_Rental_Days_Within"]
        .sum()
        .rename("Planned_Rental_Days")
    )

    inventory = (
        work.groupby("Equipment")["Total_Inventory"]
        .first()
        .rename("Total_Inventory")
    )

    result = pd.concat(
        [rental_days, inventory],
        axis=1
    ).reset_index()

    result["Available_Equipment_Days"] = (
        result["Total_Inventory"] * window_days
    )

    result["Utilisation_%"] = (
        result["Planned_Rental_Days"]
        / result["Available_Equipment_Days"]
        * 100
    )

    return result


if os.path.exists(UTILISATION_PATH):
    utilisation = pd.read_csv(UTILISATION_PATH)
    utilisation_source = "equipment_utilisation.csv"
else:
    utilisation = derive_utilisation_if_missing(cleaned)
    utilisation_source = "fallback derived from cleaned data"

if "Equipment" not in utilisation.columns:
    raise ValueError("Equipment utilisation output must contain an Equipment column.")

if "Utilisation_%" not in utilisation.columns:
    raise ValueError("Equipment utilisation output must contain Utilisation_%.")

utilisation = utilisation[
    ["Equipment", "Utilisation_%"]
].drop_duplicates("Equipment")

print("Utilisation source:", utilisation_source)
display(utilisation.sort_values("Utilisation_%", ascending=False).head(10))

## 2. Existing demand prediction input

The notebook searches the processed folder for an existing demand-prediction file.

It prefers columns such as `Predicted_Demand`, `Demand_Prediction`, `Forecast`, etc. If a prediction file is not present, a clearly labelled historical-demand proxy is used only so that the notebook remains executable; **no new ML model is trained here**.

In [ ]:
PREFERRED_DEMAND_FILES = [
    "demand_predictions.csv",
    "demand_prediction_results.csv",
    "demand_forecast.csv",
    "demand_prediction_dataset.csv",
    "demand_prediction_model_dataset.csv",
]

def first_matching_column(columns, exact_names=None, contains_all=None):
    exact_names = exact_names or []
    contains_all = contains_all or []

    lookup = {c.lower(): c for c in columns}

    for name in exact_names:
        if name.lower() in lookup:
            return lookup[name.lower()]

    for c in columns:
        cl = c.lower()
        if all(token in cl for token in contains_all):
            return c

    return None


def load_existing_demand_prediction(data_dir, preferred_files):
    candidates = []

    for name in preferred_files:
        path = os.path.join(data_dir, name)
        if os.path.exists(path):
            candidates.append(path)

    for path in sorted(glob.glob(os.path.join(data_dir, "*.csv"))):
        if path not in candidates:
            candidates.append(path)

    for path in candidates:
        try:
            temp = pd.read_csv(path)
        except Exception:
            continue

        equipment_col = first_matching_column(
            temp.columns,
            exact_names=["Equipment", "Equipment_Name", "Equipment_Type"]
        )

        prediction_col = first_matching_column(
            temp.columns,
            exact_names=[
                "Predicted_Demand",
                "Demand_Prediction",
                "PredictedDemand",
                "Forecast_Demand",
                "Future_Demand"
            ]
        )

        if prediction_col is None:
            for c in temp.columns:
                cl = c.lower()
                if (
                    "demand" in cl
                    and any(token in cl for token in ["pred", "forecast", "future"])
                    and pd.api.types.is_numeric_dtype(temp[c])
                ):
                    prediction_col = c
                    break

        if equipment_col is None or prediction_col is None:
            continue

        result = temp[[equipment_col, prediction_col]].copy()
        result.columns = ["Equipment", "Predicted_Demand"]

        # If a date/month column exists, retain the latest prediction
        # available for each equipment type.
        date_col = None

        for c in temp.columns:
            cl = c.lower()
            if any(token in cl for token in ["date", "month", "period", "forecast"]):
                parsed = pd.to_datetime(temp[c], errors="coerce")
                if parsed.notna().sum() > 0:
                    date_col = c
                    break

        if date_col is not None:
            result["_Forecast_Date"] = pd.to_datetime(
                temp[date_col],
                errors="coerce"
            )

            result = (
                result
                .sort_values("_Forecast_Date")
                .groupby("Equipment", as_index=False)
                .tail(1)
            )

        result["Predicted_Demand"] = pd.to_numeric(
            result["Predicted_Demand"],
            errors="coerce"
        )

        result = result.dropna(
            subset=["Equipment", "Predicted_Demand"]
        )

        if not result.empty:
            return result[["Equipment", "Predicted_Demand"]], os.path.basename(path)

    return None, None


demand_prediction, demand_source = load_existing_demand_prediction(
    DATA_DIR,
    PREFERRED_DEMAND_FILES
)

if demand_prediction is not None:
    print("Using existing demand prediction output:", demand_source)
else:
    # Explicit fallback for reproducibility.
    proxy_candidates = [
        "Historical_Demand_Rolling_3M",
        "Historical_Demand_Lag_1",
        "Historical_Demand_Trend_2M",
        "Historical_Demand_Rolling_6M",
    ]

    proxy_col = next(
        (
            c for c in proxy_candidates
            if c in features.columns
        ),
        None
    )

    if proxy_col is None:
        raise FileNotFoundError(
            "No existing demand prediction output was found, and "
            "no historical demand proxy is available."
        )

    proxy = features[
        ["Equipment", proxy_col]
    ].copy()

    proxy[proxy_col] = pd.to_numeric(
        proxy[proxy_col],
        errors="coerce"
    )

    demand_prediction = (
        proxy
        .dropna(subset=[proxy_col])
        .groupby("Equipment", as_index=False)[proxy_col]
        .last()
        .rename(columns={proxy_col: "Predicted_Demand"})
    )

    demand_source = f"Historical proxy: {proxy_col}"

    print(
        "WARNING: Existing demand prediction output was not found."
    )
    print(
        "Using:", demand_source
    )

display(demand_prediction.sort_values("Predicted_Demand", ascending=False).head(10))

In [ ]:
# Merge high-level utilisation and demand information.

equipment_context = pd.DataFrame({
    "Equipment": sorted(
        cleaned["Equipment"].dropna().unique()
    )
})

equipment_context = (
    equipment_context
    .merge(
        utilisation,
        on="Equipment",
        how="left"
    )
    .merge(
        demand_prediction,
        on="Equipment",
        how="left"
    )
)

# Demand pressure is scaled across equipment types.
demand_max = equipment_context["Predicted_Demand"].max()

if pd.isna(demand_max) or demand_max <= 0:
    equipment_context["Future_Demand_Pressure"] = 0.0
else:
    equipment_context["Future_Demand_Pressure"] = (
        equipment_context["Predicted_Demand"]
        .clip(lower=0)
        / demand_max
    ).clip(0, 1)

display(
    equipment_context.sort_values(
        "Future_Demand_Pressure",
        ascending=False
    )
)

## 3. Unit catalogue

`Equipment_ID` is treated as the physical unit identifier.

The allocation algorithm does not invent unit IDs. Unit records with missing `Equipment_ID` are not used as physical candidates.

In [ ]:
unit_catalog = (
    cleaned[
        cleaned["Equipment_ID"].notna()
    ]
    .sort_values(
        ["Equipment_ID", "Booking_Date", "Rental_Start_Date"]
    )
    .groupby("Equipment_ID", as_index=False)
    .tail(1)
)

UNIT_CATALOG_COLUMNS = [
    "Equipment_ID",
    "Equipment",
    "Equipment_Category",
    "Total_Inventory",
    "Equipment_Age_Years",
]

unit_catalog = unit_catalog[
    [c for c in UNIT_CATALOG_COLUMNS if c in unit_catalog.columns]
].drop_duplicates("Equipment_ID")

print("Unique physical units:", unit_catalog["Equipment_ID"].nunique())
print("Equipment types:", unit_catalog["Equipment"].nunique())
print("Inventory total:", int(unit_catalog["Total_Inventory"].sum()))

unit_count_check = (
    unit_catalog.groupby("Equipment")["Equipment_ID"]
    .nunique()
    .rename("Known_Units")
    .to_frame()
    .join(
        unit_catalog.groupby("Equipment")["Total_Inventory"]
        .first()
        .rename("Reported_Inventory")
    )
)

unit_count_check["Difference"] = (
    unit_count_check["Reported_Inventory"]
    - unit_count_check["Known_Units"]
)

display(unit_count_check)

In [ ]:
CONDITION_SCORE = {
    "Excellent": 100,
    "Good": 85,
    "Fair": 65,
    "Poor": 40,
}

def normalize_condition(value):
    if pd.isna(value):
        return "Unknown"
    return str(value).strip().title()


def condition_score(value):
    return CONDITION_SCORE.get(
        normalize_condition(value),
        55
    )

## 4. Build a time-aware unit snapshot

For a request date, only reservations with `Booking_Date <= request_date` are treated as already known reservations.

This prevents future bookings that had not yet been placed from being used as if they were known at decision time.

In [ ]:
def build_unit_snapshot(
    reference_date,
    exclude_rental_id=None
):
    reference_date = pd.Timestamp(reference_date).normalize()

    known = cleaned[
        (~cleaned["_Is_Cancelled"])
        &
        cleaned["Equipment_ID"].notna()
        &
        cleaned["Booking_Date"].notna()
        &
        (cleaned["Booking_Date"] <= reference_date)
    ].copy()

    if exclude_rental_id is not None:
        known = known[
            known["Rental_ID"].astype(str)
            != str(exclude_rental_id)
        ].copy()

    # Latest known metadata for each unit.
    latest_metadata = (
        known
        .sort_values(
            ["Equipment_ID", "Booking_Date", "Rental_Start_Date"]
        )
        .groupby("Equipment_ID", as_index=False)
        .tail(1)
    )

    latest_metadata = latest_metadata[
        [
            "Equipment_ID",
            "Equipment",
            "Equipment_Category",
            "Equipment_Condition",
            "Next_Maintenance_Due",
            "Equipment_Age_Years",
            "Total_Inventory",
        ]
    ].drop_duplicates("Equipment_ID")

    snapshot = (
        unit_catalog
        .merge(
            latest_metadata[
                [
                    "Equipment_ID",
                    "Equipment_Condition",
                    "Next_Maintenance_Due",
                    "Equipment_Age_Years",
                ]
            ],
            on="Equipment_ID",
            how="left",
            suffixes=("", "_Known")
        )
    )

    # Current occupancy at reference date.
    active = known[
        (known["Rental_Start_Date"] <= reference_date)
        &
        (known["End_Date"] > reference_date)
    ].copy()

    active_now = (
        active
        .sort_values("End_Date")
        .groupby("Equipment_ID", as_index=False)
        .tail(1)
    )

    active_map = active_now.set_index("Equipment_ID")

    snapshot["Currently_Rented"] = (
        snapshot["Equipment_ID"]
        .isin(active_now["Equipment_ID"])
    )

    snapshot["Current_Return_Date"] = snapshot["Equipment_ID"].map(
        active_map["End_Date"] if not active_map.empty else pd.Series(dtype="datetime64[ns]")
    )

    # Next scheduled booking already known at the reference date.
    upcoming = known[
        known["Rental_Start_Date"] >= reference_date
    ].copy()

    upcoming = (
        upcoming
        .sort_values("Rental_Start_Date")
        .groupby("Equipment_ID", as_index=False)
        .first()
    )

    upcoming_map = upcoming.set_index("Equipment_ID")

    snapshot["Next_Booked_Start"] = snapshot["Equipment_ID"].map(
        upcoming_map["Rental_Start_Date"]
        if not upcoming_map.empty
        else pd.Series(dtype="datetime64[ns]")
    )

    # Unit-level historical utilisation up to the reference date.
    hist = known[
        known["Rental_Start_Date"] < reference_date
    ].copy()

    if not hist.empty:
        hist["Util_End"] = hist["End_Date"].clip(
            upper=reference_date
        )

        hist["Util_Start"] = hist["Rental_Start_Date"]

        hist["Rental_Days_To_Date"] = (
            hist["Util_End"]
            - hist["Util_Start"]
        ).dt.days.clip(lower=0)

        unit_days = (
            hist.groupby("Equipment_ID")["Rental_Days_To_Date"]
            .sum()
        )

        observation_days = max(
            (reference_date - cleaned["Rental_Start_Date"].min()).days,
            1
        )

        snapshot["Unit_Rental_Days_To_Date"] = (
            snapshot["Equipment_ID"]
            .map(unit_days)
            .fillna(0)
        )

        snapshot["Current_Unit_Utilisation_%"] = (
            snapshot["Unit_Rental_Days_To_Date"]
            / observation_days
            * 100
        ).clip(0, 100)
    else:
        snapshot["Unit_Rental_Days_To_Date"] = 0.0
        snapshot["Current_Unit_Utilisation_%"] = 0.0

    snapshot["Reference_Date"] = reference_date

    return snapshot

## 5. Candidate generation

Availability is treated as a **hard constraint**:

- correct equipment type
- not currently rented
- no known reservation overlaps the requested rental period
- known maintenance due date must not fall inside the requested period

`STRICT_MAINTENANCE = False` is the default because the dataset contains incomplete maintenance information. Set it to `True` for a stricter operational policy.

In [ ]:
STRICT_MAINTENANCE = False

def get_future_demand_pressure(equipment):
    row = equipment_context[
        equipment_context["Equipment"] == equipment
    ]

    if row.empty:
        return 0.0, np.nan

    pressure = float(
        row["Future_Demand_Pressure"].iloc[0]
    )

    predicted = float(
        row["Predicted_Demand"].iloc[0]
    ) if pd.notna(row["Predicted_Demand"].iloc[0]) else np.nan

    return pressure, predicted


def get_candidates(
    equipment,
    request_start,
    rental_duration_days,
    strict_maintenance=STRICT_MAINTENANCE,
    exclude_rental_id=None
):
    request_start = pd.Timestamp(
        request_start
    ).normalize()

    rental_duration_days = int(rental_duration_days)

    if rental_duration_days <= 0:
        raise ValueError("Rental duration must be greater than 0.")

    request_end = (
        request_start
        + pd.Timedelta(days=rental_duration_days)
    )

    snapshot = build_unit_snapshot(
        request_start,
        exclude_rental_id=exclude_rental_id
    )

    same_type = snapshot[
        snapshot["Equipment"] == equipment
    ].copy()

    # Current availability.
    candidates = same_type[
        ~same_type["Currently_Rented"]
    ].copy()

    if candidates.empty:
        return candidates, snapshot, request_end

    # Known booking conflict over the requested interval.
    known = cleaned[
        (~cleaned["_Is_Cancelled"])
        &
        cleaned["Equipment_ID"].notna()
        &
        cleaned["Booking_Date"].notna()
        &
        (cleaned["Booking_Date"] <= request_start)
        &
        (cleaned["Equipment"] == equipment)
    ].copy()

    if exclude_rental_id is not None:
        known = known[
            known["Rental_ID"].astype(str)
            != str(exclude_rental_id)
        ].copy()

    conflict_counts = {}

    for equipment_id, group in known.groupby("Equipment_ID"):
        overlap = (
            (group["Rental_Start_Date"] < request_end)
            &
            (group["End_Date"] > request_start)
        )

        conflict_counts[equipment_id] = int(overlap.sum())

    candidates["Known_Booking_Conflicts"] = (
        candidates["Equipment_ID"]
        .map(conflict_counts)
        .fillna(0)
        .astype(int)
    )

    candidates = candidates[
        candidates["Known_Booking_Conflicts"] == 0
    ].copy()

    if candidates.empty:
        return candidates, snapshot, request_end

    # Maintenance conflict.
    candidates["Maintenance_Conflict"] = (
        candidates["Next_Maintenance_Due"].notna()
        &
        (candidates["Next_Maintenance_Due"] >= request_start)
        &
        (candidates["Next_Maintenance_Due"] < request_end)
    )

    candidates = candidates[
        ~candidates["Maintenance_Conflict"]
    ].copy()

    if strict_maintenance:
        candidates = candidates[
            candidates["Next_Maintenance_Due"].isna()
            |
            (candidates["Next_Maintenance_Due"] >= request_start)
        ].copy()

    if candidates.empty:
        return candidates, snapshot, request_end

    # Free window until the next already-known reservation.
    candidates["Availability_Window_Days"] = np.where(
        candidates["Next_Booked_Start"].notna(),
        (
            candidates["Next_Booked_Start"] - request_start
        ).dt.days,
        3650
    )

    # Maintenance horizon.
    candidates["Maintenance_Horizon_Days"] = np.where(
        candidates["Next_Maintenance_Due"].notna(),
        (
            candidates["Next_Maintenance_Due"] - request_start
        ).dt.days,
        np.nan
    )

    candidates["Condition_Normalized"] = (
        candidates["Equipment_Condition"]
        .map(normalize_condition)
    )

    pressure, predicted_demand = get_future_demand_pressure(
        equipment
    )

    candidates["Predicted_Future_Demand"] = predicted_demand
    candidates["Future_Demand_Pressure"] = pressure

    return candidates, snapshot, request_end

## 6. Suitability scoring

The weights are a transparent decision policy rather than a learned ML model.

| Factor | Weight |
|---|---:|
| Availability | 15% |
| Condition | 20% |
| Maintenance status | 20% |
| Availability window / expected next commitment | 15% |
| Rental-duration fit | 10% |
| Current unit utilisation | 15% |
| Future demand | 5% |

Availability remains a hard filter, so every scored candidate is actually available for the requested period.

In [ ]:
WEIGHTS = {
    "Availability_Score": 0.15,
    "Condition_Score": 0.20,
    "Maintenance_Score": 0.20,
    "Availability_Window_Score": 0.15,
    "Duration_Fit_Score": 0.10,
    "Current_Utilisation_Score": 0.15,
    "Future_Demand_Score": 0.05,
}


def calculate_maintenance_score(
    maintenance_due,
    request_start,
    request_end
):
    if pd.isna(maintenance_due):
        return 55.0

    maintenance_due = pd.Timestamp(
        maintenance_due
    ).normalize()

    request_start = pd.Timestamp(
        request_start
    ).normalize()

    request_end = pd.Timestamp(
        request_end
    ).normalize()

    if maintenance_due < request_start:
        return 35.0

    if maintenance_due < request_end:
        return 0.0

    days_after_rental = (
        maintenance_due - request_end
    ).days

    if days_after_rental >= 30:
        return 100.0

    return 85.0


def calculate_duration_fit(
    maintenance_due,
    request_start,
    rental_duration_days
):
    if pd.isna(maintenance_due):
        return 60.0

    horizon = (
        pd.Timestamp(maintenance_due).normalize()
        - pd.Timestamp(request_start).normalize()
    ).days

    if horizon < 0:
        return 0.0

    if horizon < rental_duration_days:
        return 0.0

    # 70 at the exact required duration, reaching 100
    # when there is at least an additional rental-duration buffer.
    extra_buffer_ratio = (
        (horizon - rental_duration_days)
        / max(rental_duration_days, 1)
    )

    return float(
        np.clip(
            70 + 30 * min(extra_buffer_ratio, 1),
            0,
            100
        )
    )


def score_candidates(
    candidates,
    request_start,
    request_end,
    rental_duration_days
):
    if candidates.empty:
        return candidates.copy()

    scored = candidates.copy()

    scored["Availability_Score"] = 100.0

    scored["Condition_Score"] = (
        scored["Equipment_Condition"]
        .apply(condition_score)
    )

    scored["Maintenance_Score"] = scored.apply(
        lambda row: calculate_maintenance_score(
            row["Next_Maintenance_Due"],
            request_start,
            request_end
        ),
        axis=1
    )

    scored["Availability_Window_Score"] = np.minimum(
        100,
        40
        + 60
        * np.minimum(
            scored["Availability_Window_Days"]
            / max(3 * rental_duration_days, 1),
            1
        )
    )

    scored["Duration_Fit_Score"] = scored.apply(
        lambda row: calculate_duration_fit(
            row["Next_Maintenance_Due"],
            request_start,
            rental_duration_days
        ),
        axis=1
    )

    base_utilisation_score = (
        100
        - scored["Current_Unit_Utilisation_%"].fillna(0)
    ).clip(0, 100)

    # High future demand makes preserving under-utilised units
    # more important. This creates a demand-aware utilisation score.
    demand_multiplier = (
        1
        + 0.50 * scored["Future_Demand_Pressure"].fillna(0)
    )

    scored["Current_Utilisation_Score"] = np.minimum(
        100,
        base_utilisation_score * demand_multiplier
    )

    scored["Future_Demand_Score"] = (
        100
        * (
            1
            - scored["Future_Demand_Pressure"].fillna(0)
        )
    )

    scored["Suitability_Score"] = 0.0

    for component, weight in WEIGHTS.items():
        scored["Suitability_Score"] += (
            scored[component] * weight
        )

    scored["Suitability_Score"] = (
        scored["Suitability_Score"]
        .clip(0, 100)
        .round(2)
    )

    scored = scored.sort_values(
        [
            "Suitability_Score",
            "Condition_Score",
            "Maintenance_Score",
            "Availability_Window_Days",
            "Current_Unit_Utilisation_%"
        ],
        ascending=[
            False,
            False,
            False,
            False,
            True
        ]
    ).reset_index(drop=True)

    scored["Rank"] = np.arange(
        1,
        len(scored) + 1
    )

    return scored

## 7. Recommendation engine

The engine returns:

- recommended unit
- suitability score
- condition
- maintenance status
- availability window
- current unit utilisation
- predicted future demand
- explanation

When there is no valid unit, it does **not** falsely recommend an unavailable unit. It instead reports the soonest known availability options.

In [ ]:
def maintenance_label(maintenance_due, request_start, request_end):
    if pd.isna(maintenance_due):
        return "Unknown"

    maintenance_due = pd.Timestamp(
        maintenance_due
    ).normalize()

    request_start = pd.Timestamp(
        request_start
    ).normalize()

    request_end = pd.Timestamp(
        request_end
    ).normalize()

    if maintenance_due < request_start:
        return "Overdue"

    if maintenance_due < request_end:
        return "Due During Rental"

    if maintenance_due <= request_end + pd.Timedelta(days=30):
        return "Due Soon"

    return "Not Due"


def build_explanation(row, request_start, request_end):
    condition = normalize_condition(
        row["Equipment_Condition"]
    )

    maintenance = maintenance_label(
        row["Next_Maintenance_Due"],
        request_start,
        request_end
    )

    window = row["Availability_Window_Days"]

    if window >= 3650:
        window_text = "no known upcoming booking"
    else:
        window_text = f"{int(window)} day(s) before the next known booking"

    utilisation = row["Current_Unit_Utilisation_%"]

    if row["Future_Demand_Pressure"] >= 0.75:
        demand_text = "future demand is high, so a less-utilised unit is preferred"
    elif row["Future_Demand_Pressure"] >= 0.40:
        demand_text = "future demand is moderate"
    else:
        demand_text = "future demand pressure is relatively low"

    return (
        f"{condition} condition; maintenance status: {maintenance}; "
        f"{window_text}; current unit utilisation is "
        f"{utilisation:.1f}%; {demand_text}."
    )


def next_available_options(
    equipment,
    request_start,
    exclude_rental_id=None
):
    snapshot = build_unit_snapshot(
        request_start,
        exclude_rental_id=exclude_rental_id
    )

    options = snapshot[
        snapshot["Equipment"] == equipment
    ].copy()

    if options.empty:
        return options

    options["Expected_Available_Date"] = np.where(
        options["Currently_Rented"],
        options["Current_Return_Date"],
        pd.Timestamp(request_start)
    )

    options["Expected_Available_Date"] = pd.to_datetime(
        options["Expected_Available_Date"],
        errors="coerce"
    )

    options = options.sort_values(
        [
            "Expected_Available_Date",
            "Condition_Score"
            if "Condition_Score" in options.columns
            else "Equipment_ID"
        ],
        ascending=[True, False]
        if "Condition_Score" in options.columns
        else [True]
    )

    return options[
        [
            "Equipment_ID",
            "Equipment",
            "Equipment_Condition",
            "Current_Return_Date",
            "Expected_Available_Date"
        ]
    ].head(5)


def recommend_equipment(
    equipment,
    rental_duration_days,
    request_start,
    strict_maintenance=STRICT_MAINTENANCE,
    exclude_rental_id=None,
    return_candidates=False
):
    request_start = pd.Timestamp(
        request_start
    ).normalize()

    candidates, snapshot, request_end = get_candidates(
        equipment=equipment,
        request_start=request_start,
        rental_duration_days=rental_duration_days,
        strict_maintenance=strict_maintenance,
        exclude_rental_id=exclude_rental_id
    )

    if candidates.empty:
        result = {
            "Recommendation_Status": "No suitable unit currently available",
            "Equipment": equipment,
            "Request_Start_Date": request_start,
            "Request_End_Date": request_end,
            "Rental_Duration_Days": rental_duration_days,
            "Recommended_Equipment_ID": None,
            "Suitability_Score": None,
            "Reason": (
                "No unit satisfies the current availability "
                "and maintenance-period constraints."
            ),
            "Candidates": candidates,
            "Next_Available_Options": next_available_options(
                equipment,
                request_start,
                exclude_rental_id=exclude_rental_id
            )
        }
        return result

    scored = score_candidates(
        candidates,
        request_start=request_start,
        request_end=request_end,
        rental_duration_days=rental_duration_days
    )

    best = scored.iloc[0].copy()

    result = {
        "Recommendation_Status": "Recommended",
        "Equipment": equipment,
        "Request_Start_Date": request_start,
        "Request_End_Date": request_end,
        "Rental_Duration_Days": rental_duration_days,
        "Recommended_Equipment_ID": best["Equipment_ID"],
        "Suitability_Score": float(best["Suitability_Score"]),
        "Condition": normalize_condition(best["Equipment_Condition"]),
        "Maintenance_Status": maintenance_label(
            best["Next_Maintenance_Due"],
            request_start,
            request_end
        ),
        "Availability_Window_Days": float(
            best["Availability_Window_Days"]
        ),
        "Current_Unit_Utilisation_%": float(
            best["Current_Unit_Utilisation_%"]
        ),
        "Predicted_Future_Demand": (
            float(best["Predicted_Future_Demand"])
            if pd.notna(best["Predicted_Future_Demand"])
            else np.nan
        ),
        "Reason": build_explanation(
            best,
            request_start,
            request_end
        ),
        "Candidates": scored,
        "Next_Available_Options": pd.DataFrame()
    }

    return result

## 8. Demo request

The PRD example uses an Oxygen Concentrator for 12 days. Change the three values below for dashboard testing.

In [ ]:
REQUEST_EQUIPMENT = "Oxygen Concentrator"
REQUEST_START_DATE = pd.Timestamp(
    cleaned["Rental_Start_Date"].max()
)
REQUEST_DURATION_DAYS = 12

demo = recommend_equipment(
    equipment=REQUEST_EQUIPMENT,
    rental_duration_days=REQUEST_DURATION_DAYS,
    request_start=REQUEST_START_DATE,
    strict_maintenance=STRICT_MAINTENANCE
)

print("Recommendation Status:", demo["Recommendation_Status"])

if demo["Recommendation_Status"] == "Recommended":
    print("Recommended Equipment:", demo["Recommended_Equipment_ID"])
    print("Suitability Score:", f"{demo['Suitability_Score']:.2f}%")
    print("Condition:", demo["Condition"])
    print("Maintenance:", demo["Maintenance_Status"])
    print(
        "Availability Window:",
        f"{demo['Availability_Window_Days']:.0f} day(s)"
    )
    print(
        "Current Unit Utilisation:",
        f"{demo['Current_Unit_Utilisation_%']:.2f}%"
    )
    print("Reason:", demo["Reason"])
else:
    print("Reason:", demo["Reason"])
    display(demo["Next_Available_Options"])

In [ ]:
# Candidate ranking for the demo request.

if demo["Recommendation_Status"] == "Recommended":
    candidate_view = demo["Candidates"][
        [
            "Rank",
            "Equipment_ID",
            "Equipment_Condition",
            "Suitability_Score",
            "Maintenance_Score",
            "Availability_Window_Score",
            "Duration_Fit_Score",
            "Current_Utilisation_Score",
            "Future_Demand_Score",
            "Current_Unit_Utilisation_%",
            "Availability_Window_Days",
            "Next_Booked_Start",
            "Next_Maintenance_Due",
        ]
    ].copy()

    display(candidate_view.head(10))

In [ ]:
# Visual score comparison for top candidates.

if demo["Recommendation_Status"] == "Recommended":
    top = demo["Candidates"].head(5).copy()

    plt.figure(figsize=(10, 5))
    plt.bar(
        top["Equipment_ID"].astype(str),
        top["Suitability_Score"]
    )
    plt.title("Top Candidate Suitability Scores")
    plt.xlabel("Equipment Unit")
    plt.ylabel("Suitability Score (%)")
    plt.ylim(0, 100)
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## 9. Test the recommendation engine on several equipment types

In [ ]:
example_equipment = sorted(
    cleaned["Equipment"]
    .dropna()
    .unique()
)[:8]

demo_results = []

for equipment in example_equipment:
    result = recommend_equipment(
        equipment=equipment,
        rental_duration_days=7,
        request_start=REQUEST_START_DATE,
        strict_maintenance=STRICT_MAINTENANCE
    )

    demo_results.append({
        "Equipment": equipment,
        "Status": result["Recommendation_Status"],
        "Recommended_Equipment_ID": result.get(
            "Recommended_Equipment_ID"
        ),
        "Suitability_Score": result.get(
            "Suitability_Score"
        ),
        "Condition": result.get("Condition"),
        "Maintenance_Status": result.get(
            "Maintenance_Status"
        ),
        "Reason": result.get("Reason"),
    })

demo_results_df = pd.DataFrame(demo_results)

display(demo_results_df)

## 10. Constraint validation

Every recommendation should satisfy:

1. Correct equipment type
2. Unit is not currently rented
3. No known booking overlaps the requested period
4. No known maintenance due date falls inside the rental
5. Recommendation score is between 0 and 100

In [ ]:
def validate_recommendation(result):
    if result["Recommendation_Status"] != "Recommended":
        return {
            "Equipment_Type_OK": None,
            "Availability_OK": None,
            "Booking_Conflict_OK": None,
            "Maintenance_OK": None,
            "Score_OK": None,
            "Overall_OK": False,
        }

    candidates = result["Candidates"]

    best_id = result["Recommended_Equipment_ID"]

    selected = candidates[
        candidates["Equipment_ID"] == best_id
    ].iloc[0]

    equipment_ok = (
        selected["Equipment"] == result["Equipment"]
    )

    availability_ok = (
        not bool(selected["Currently_Rented"])
    )

    booking_ok = (
        int(selected["Known_Booking_Conflicts"]) == 0
    )

    maintenance_ok = (
        not bool(selected["Maintenance_Conflict"])
    )

    score_ok = (
        0 <= float(selected["Suitability_Score"]) <= 100
    )

    overall = all([
        equipment_ok,
        availability_ok,
        booking_ok,
        maintenance_ok,
        score_ok,
    ])

    return {
        "Equipment_Type_OK": equipment_ok,
        "Availability_OK": availability_ok,
        "Booking_Conflict_OK": booking_ok,
        "Maintenance_OK": maintenance_ok,
        "Score_OK": score_ok,
        "Overall_OK": overall,
    }


demo_validation = validate_recommendation(demo)

print("Validation:")
for k, v in demo_validation.items():
    print(f"{k}: {v}")

## 11. Historical recommendation evaluation

There is no ground-truth label saying which physical unit was the "best" unit. Therefore, exact-unit accuracy is **not** treated as the primary metric.

Instead, the evaluation checks:

- recommendation availability validity
- equipment-type correctness
- maintenance-period safety
- absence of known reservation conflicts
- exact historical unit match as a secondary diagnostic only

The target rental itself is excluded from the decision snapshot so that evaluation simulates the moment before that customer reservation was created.

In [ ]:
def evaluate_one_historical_request(row):
    result = recommend_equipment(
        equipment=row["Equipment"],
        rental_duration_days=int(row["Rental_Duration_Days"]),
        request_start=row["Rental_Start_Date"],
        strict_maintenance=False,
        exclude_rental_id=row["Rental_ID"]
    )

    if result["Recommendation_Status"] != "Recommended":
        return {
            "Rental_ID": row["Rental_ID"],
            "Equipment": row["Equipment"],
            "Recommended": False,
            "Recommended_Equipment_ID": None,
            "Actual_Historical_Equipment_ID": row["Equipment_ID"],
            "Equipment_Type_OK": False,
            "Known_Availability_OK": False,
            "Maintenance_OK": False,
            "Suitability_Score": np.nan,
            "Exact_Unit_Match": False,
        }

    selected_id = result["Recommended_Equipment_ID"]
    candidates = result["Candidates"]

    selected = candidates[
        candidates["Equipment_ID"] == selected_id
    ].iloc[0]

    return {
        "Rental_ID": row["Rental_ID"],
        "Equipment": row["Equipment"],
        "Recommended": True,
        "Recommended_Equipment_ID": selected_id,
        "Actual_Historical_Equipment_ID": row["Equipment_ID"],
        "Equipment_Type_OK": (
            selected["Equipment"] == row["Equipment"]
        ),
        "Known_Availability_OK": (
            not bool(selected["Currently_Rented"])
            and int(selected["Known_Booking_Conflicts"]) == 0
        ),
        "Maintenance_OK": (
            not bool(selected["Maintenance_Conflict"])
        ),
        "Suitability_Score": float(
            selected["Suitability_Score"]
        ),
        "Exact_Unit_Match": (
            str(selected_id) == str(row["Equipment_ID"])
        ),
    }

In [ ]:
evaluation_pool = cleaned[
    (~cleaned["_Is_Cancelled"])
    &
    cleaned["Equipment_ID"].notna()
    &
    cleaned["Rental_Start_Date"].notna()
].copy()

# Keep evaluation manageable while covering multiple equipment types.
sampled_groups = []

for equipment_name, group in evaluation_pool.groupby("Equipment"):
    sampled_groups.append(
        group.sample(
            n=min(5, len(group)),
            random_state=42
        )
    )

if sampled_groups:
    evaluation_pool = pd.concat(
        sampled_groups,
        ignore_index=True
    )
else:
    evaluation_pool = pd.DataFrame()

evaluation_rows = []

for _, row in evaluation_pool.iterrows():
    evaluation_rows.append(
        evaluate_one_historical_request(row)
    )

evaluation_df = pd.DataFrame(evaluation_rows)

display(evaluation_df.head(20))

In [ ]:
# Evaluation summary

if evaluation_df.empty:
    print("No historical evaluation cases were generated.")
else:
    summary = {
        "Total_Test_Cases": len(evaluation_df),
        "Recommendation_Rate_%": (
            evaluation_df["Recommended"].mean() * 100
        ),
        "Equipment_Type_Validity_%": (
            evaluation_df.loc[
                evaluation_df["Recommended"],
                "Equipment_Type_OK"
            ].mean() * 100
            if evaluation_df["Recommended"].any()
            else np.nan
        ),
        "Known_Availability_Constraint_%": (
            evaluation_df.loc[
                evaluation_df["Recommended"],
                "Known_Availability_OK"
            ].mean() * 100
            if evaluation_df["Recommended"].any()
            else np.nan
        ),
        "Maintenance_Safety_%": (
            evaluation_df.loc[
                evaluation_df["Recommended"],
                "Maintenance_OK"
            ].mean() * 100
            if evaluation_df["Recommended"].any()
            else np.nan
        ),
        "Exact_Unit_Match_%_Secondary": (
            evaluation_df.loc[
                evaluation_df["Recommended"],
                "Exact_Unit_Match"
            ].mean() * 100
            if evaluation_df["Recommended"].any()
            else np.nan
        ),
        "Overall_Constraint_Satisfaction_%": (
            (
                evaluation_df.loc[
                    evaluation_df["Recommended"],
                    [
                        "Equipment_Type_OK",
                        "Known_Availability_OK",
                        "Maintenance_OK",
                    ]
                ]
                .all(axis=1)
                .mean()
                * 100
            )
            if evaluation_df["Recommended"].any()
            else np.nan
        ),
    }

    evaluation_summary = pd.DataFrame(
        [summary]
    )

    display(evaluation_summary.T)

## 12. Save outputs

These files can be consumed by the dashboard/backend later.

- `smart_allocation_demo_results.csv` - sample recommendations
- `smart_allocation_candidate_scores.csv` - ranked candidates for the demo request
- `smart_allocation_evaluation.csv` - historical evaluation cases
- `smart_allocation_evaluation_summary.csv` - evaluation metrics

In [ ]:
demo_output_path = os.path.join(
    DATA_DIR,
    "smart_allocation_demo_results.csv"
)

candidate_output_path = os.path.join(
    DATA_DIR,
    "smart_allocation_candidate_scores.csv"
)

evaluation_output_path = os.path.join(
    DATA_DIR,
    "smart_allocation_evaluation.csv"
)

evaluation_summary_path = os.path.join(
    DATA_DIR,
    "smart_allocation_evaluation_summary.csv"
)

demo_results_df.to_csv(
    demo_output_path,
    index=False
)

if demo["Recommendation_Status"] == "Recommended":
    demo["Candidates"].to_csv(
        candidate_output_path,
        index=False
    )
else:
    pd.DataFrame().to_csv(
        candidate_output_path,
        index=False
    )

evaluation_df.to_csv(
    evaluation_output_path,
    index=False
)

if "evaluation_summary" in globals():
    evaluation_summary.to_csv(
        evaluation_summary_path,
        index=False
    )

print("Saved:")
print(" -", demo_output_path)
print(" -", candidate_output_path)
print(" -", evaluation_output_path)
if "evaluation_summary" in globals():
    print(" -", evaluation_summary_path)

## Final module summary

### Decision flow

**Customer request → availability filter → maintenance/booking constraints → candidate scoring → ranking → recommendation + explanation**

### What this module adds

- unit-level equipment recommendation
- transparent suitability score
- maintenance-aware allocation
- duration-aware allocation
- unit-level utilisation balancing
- future-demand-aware allocation
- explainable recommendation reasons
- historical constraint-based evaluation

### Architecture position

This is a decision layer on top of the already-built **Demand Prediction** and **Equipment Utilisation** modules. It does not create a duplicate ML model.

In [ ]:
print("=" * 60)
print("SMART EQUIPMENT ALLOCATION MODULE COMPLETE")
print("=" * 60)
print("Demand source:", demand_source)
print("Strict maintenance mode:", STRICT_MAINTENANCE)
print("Physical units available to allocator:", unit_catalog["Equipment_ID"].nunique())
print("Equipment types:", cleaned["Equipment"].nunique())
print("Demo equipment:", REQUEST_EQUIPMENT)
print("Demo duration:", REQUEST_DURATION_DAYS, "days")
print("Demo request date:", REQUEST_START_DATE.date())
print("Recommendation status:", demo["Recommendation_Status"])

if demo["Recommendation_Status"] == "Recommended":
    print("Recommended unit:", demo["Recommended_Equipment_ID"])
    print("Suitability:", f"{demo['Suitability_Score']:.2f}%")